# Step 4: Real Spatial Data (No Fake Sensors)

## Why this notebook exists

The earlier pipeline built a **fake triangle** around each reef and added random ±0.2°C noise.
That was fine for demos — but the model was not learning real spatial differences.

Here we use the **5 real satellite locations** already in `sliot_dataset/`:

| Location | Approx coords |
|----------|---------------|
| Hikkaduwa | 6.125°N, 80.075°E |
| Kalpitiya | 8.375°N, 79.725°E |
| Passikudah | 7.925°N, 81.575°E |
| South East | 6.175°N, 81.475°E |
| Trincomalee | 8.725°N, 81.175°E |

**What you get:** clean SST + DHW tables, scalers, and **time / location hold-out splits** ready for training.

## What changed vs the old approach

```
OLD (demo)                         NEW (this notebook)
-------------                      -------------------
1 SST series                       5 real reef sites
+ triangle offsets                 real lat/lon only
+ random ±0.2°C noise              no synthetic noise
shuffled / prefix split            hold out future dates
                                   + leave one location out
```

## Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from prepare_data import (
    LOCATIONS,
    load_all_locations,
    prepare_and_save,
    time_holdout_split,
    location_holdout_split,
)

print("Libraries loaded")
print("Locations:", LOCATIONS)

## Load real multi-location SST + DHW

Each site already has `sst_full.csv` and `dhw_full.csv`.
We merge them on time — **no triangle offsets, no random noise**.

In [ ]:
df = load_all_locations()

print("Combined shape:", df.shape)
print("Date range:", df["time"].min(), "→", df["time"].max())
print("\nRows per location:")
print(df.groupby("location").size())
df.head()

## Peek at real spatial differences

If the sites were just copies of one series, mean temperatures would be almost identical.
Real satellite points should differ by location.

In [ ]:
summary = (
    df.groupby("location")
    .agg(
        lat=("latitude", "first"),
        lon=("longitude", "first"),
        sst_mean=("analysed_sst", "mean"),
        sst_std=("analysed_sst", "std"),
        dhw_mean=("degree_heating_week", "mean"),
        dhw_max=("degree_heating_week", "max"),
    )
    .round(3)
)
summary

## Visualize SST time series per reef

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for loc, g in df.groupby("location"):
    # Plot a weekly downsample so the chart stays readable
    g2 = g.set_index("time")["analysed_sst"].resample("7D").mean()
    ax.plot(g2.index, g2.values, label=loc, linewidth=1.5)

ax.set_title("Weekly-mean SST at 5 real Sri Lanka reef sites", fontsize=14, fontweight="bold")
ax.set_ylabel("Temperature (°C)")
ax.set_xlabel("Time")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Hold-out strategy (important!)

A shuffled 80/20 split **leaks the future into training**.

We do two tougher checks instead:

1. **Time hold-out** — train on older dates, validate/test on the newest dates
2. **Location hold-out** — leave one reef (default: Trincomalee) completely out of training

In [ ]:
train_t, val_t, test_t = time_holdout_split(df, val_frac=0.1, test_frac=0.1)
train_loc, holdout_loc = location_holdout_split(df, holdout_location="trinco")

print("TIME HOLD-OUT")
print(f"  train : {len(train_t):5d} rows | {train_t['time'].min().date()} → {train_t['time'].max().date()}")
print(f"  val   : {len(val_t):5d} rows | {val_t['time'].min().date()} → {val_t['time'].max().date()}")
print(f"  test  : {len(test_t):5d} rows | {test_t['time'].min().date()} → {test_t['time'].max().date()}")
print()
print("LOCATION HOLD-OUT")
print(f"  train sites : {sorted(train_loc['location'].unique())}")
print(f"  holdout     : {sorted(holdout_loc['location'].unique())} ({len(holdout_loc)} rows)")

## Save everything for training

This writes the files that `02_pinn_model.ipynb` expects:

- `X_train.npy`, `y_train.npy`
- `X_val.npy`, `y_val.npy`, `X_test.npy`, `y_test.npy`
- `X_loc_holdout.npy`, `y_loc_holdout.npy`
- `scalers.pkl`, `sensor_info.pkl`
- `dataset/real_locations_data.csv` + split CSVs

In [ ]:
summary = prepare_and_save(holdout_location="trinco", val_frac=0.1, test_frac=0.1)

print("\nSaved artefacts:")
for f in [
    "X_train.npy", "y_train.npy",
    "X_val.npy", "y_val.npy",
    "X_test.npy", "y_test.npy",
    "X_loc_holdout.npy", "y_loc_holdout.npy",
    "scalers.pkl", "sensor_info.pkl",
    "dataset/real_locations_data.csv",
]:
    ok = os.path.exists(f)
    print(f"  [{'OK' if ok else 'MISSING'}] {f}")

## Summary

You now have **real multi-site data** with proper hold-outs.

**Next steps:**
1. Run `05_estimate_advection.ipynb` (optional but recommended for PDE physics)
2. Re-train with `02_pinn_model.ipynb`
3. Evaluate with `06_evaluation.ipynb`